In [0]:
%sql
-- =============================================================
-- Projet Retail Azure - Setup Unity Catalog (idempotent)
-- Prérequis : storage credential cred_ac_databricks_poc
--             (Access Connector ac-databricks-poc, rôle Storage Blob Data Contributor
--              sur le storage stdbxpocazed2026)
-- =============================================================

-- 1. External location sur le container retail
CREATE EXTERNAL LOCATION IF NOT EXISTS ext_retail
URL 'abfss://retail@stdbxpocazed2026.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL cred_ac_databricks_poc)
COMMENT 'Projet Retail - lake bronze/silver/gold';

-- 2. Catalogue du projet
--    Stockage managé par défaut dans _managed/ pour ne pas chevaucher le volume bronze/
CREATE CATALOG IF NOT EXISTS retail
MANAGED LOCATION 'abfss://retail@stdbxpocazed2026.dfs.core.windows.net/_managed'
COMMENT 'Projet Retail Azure (ADF + Databricks + Power BI)';

-- 3. Schémas Medallion
CREATE SCHEMA IF NOT EXISTS retail.bronze
COMMENT 'Fichiers bruts déposés par ADF';

CREATE SCHEMA IF NOT EXISTS retail.silver
MANAGED LOCATION 'abfss://retail@stdbxpocazed2026.dfs.core.windows.net/silver'
COMMENT 'Données nettoyées et conformes';

CREATE SCHEMA IF NOT EXISTS retail.gold
MANAGED LOCATION 'abfss://retail@stdbxpocazed2026.dfs.core.windows.net/gold'
COMMENT 'Modèle en étoile pour Power BI';

-- 4. Volume externe sur les fichiers Bronze (écrits par ADF, lus par Databricks)
CREATE EXTERNAL VOLUME IF NOT EXISTS retail.bronze.raw
LOCATION 'abfss://retail@stdbxpocazed2026.dfs.core.windows.net/bronze'
COMMENT 'Fichiers bruts ADF (SQL en Parquet, API en JSON)';

-- 5. Vérification
LIST '/Volumes/retail/bronze/raw/';